Example

In [1]:
import torch
import torch.nn as nn

In [2]:
b, num_tokens, d_in = (1,3,6)
x = torch.tensor([[
[1.0, 2.0, 3.0, 4.0, 5.0, 6.0], # 1st token
[6.0, 5.0, 4.0, 3.0, 2.0, 1.0], # 2nd token
[1.0, 1.0, 1.0, 1.0, 1.0, 1.0] # 3rd token
]])

# batch = 1, num_tokens = 3 and d_in = 6 
# forward method of the multi attention class always starts with 3 dimensions

* Step 1: Start with the input (b, num_tokens, d_in = (1,3,6)) \
* Step 2: Decide d_out and num_heads --> d_out = 6 and num_heads = 2. Thererfore the dimension of each attention head (head_dim)  ==> 6/2 = 3 \
* Step 3: Intialise trainable weight matrices for key, query and value (Wq, Wv, Wk). Dimensions of matrices are d_in and d_out. All the values inside the matrices would are random and later through back propagation we would get the optimised values for the matrices to predict the next token correctly. \
* Step 4: Calcualte keys, values and queries matrix(input * Wk, input * Wv, input * Wq) --> keys(1x3x6), Queries(1x3x6) and values(1x3x6). The dimensions of the queries, keys and values are 1x3x6 where the first two (1x3) remains the same but the same dimension(6) is d_out.
* Step 5: Unroll last dimension of keys, values and queries to include num_heads and head_dim. (b, num_tokens, d_out) --> (b, num_tokens, num_heads, head_dim)
* Step 6: Group the matrices by the number of heads ==> (b, num_tokens, num_heads,heads_dim) --> (b, num_heads, num_tokens, head_dim)
(1,3,2,3) --> (1, 2, 3, 3). We are changing the grouping by token to grouping by head. This is done as it is easier to multiply. 
This splits the head so that we can see different copies of the information for the tokens.
* Step 7: Find attention scores --> Queries * keys.T(2,3). rows ==> num_tokens and columns ==> head_dim. Therefore we are taking transpose of these two. [[Q1 * K1.T], [Q2 * K2.T]]. Q1 ==>(1, 2, 3, 3)[b, num_heads, num_tokens, head_dim], K1.T ==>(1, 2, 3, 3) [b, num_heads, head_dim, num_tokens] => (b, num_heads, num_tokens, num_tokens) (1,2,3,3) --> dimensions of the attention scores. Taking the dot product for each head.
* Step 8: Finding attention scores. Mask the attention scores to implement causal attention. Divide by (head_dim)^0.5 --> (d_out / num_heads)^0.5 ==> (6/2)^0.5.
* Step 9: Finding context vector = Attention weights(b, num_heads, num_tokens, num_tokens) * Values (b, num_heads, num_tokens, head_dim) ==>
(b, num_heads, num_tokens, head_dim)
* Step 10: Adding context vectors ==> (b, num_heads, num_tokens, head_dim) --> (b, num_tokens, num_heads, head_dim). Changing to grouping by tokens. To merge the context vector for head 1 and head 2, we need to make the grouping by tokens. \
The final resultant matrix vector would be (1, 3, 6) -> [b, num_tokens, d_out]
    

In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out,context_length, dropout, num_heads, qkv_bias = False):
        super().__init__()
        assert(d_out % num_heads ==0), \
            "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # reduce the projection dimension to match desired output dim

        # Intialising the trainable value, key and query matrices. Intialised through a linear layer of neural network and the bias term is zero.
        self.W_query = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(d_in, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal = 1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        
        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        values = self.W_value(x) # Shape: (b, num_tokens, d_out)
        queries=self.W_query(x) # Shape: (b, num_tokens, d_out)

        # We split the matrix by adding a num_heads dimension
        # unroll last dimension: (b, num_tokens, d_out) --> (b, num_tokens, num_heads, heads_dim) 
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
    
        # Transpose: (b, num_tokens, num_heads, heads_dim) --> (b, num_heads, num_tokens, heads_dim)
        keys = keys.transpose(1,2) #taking the transpose of num_tokens and num_heads interchanges it to num_heads & num_tokens
        queries = queries.transpose(1,2)
        values= values.transpose(1,2)

        # Compute scaled dot-product attention with a causal mask
        attn_scores = queries @ keys.transpose(2,3)

        # Orginial mask truncated to the number of tokens converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill the attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5 , dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1,2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec
    
"""
Step 1: Reduce the projection dim to match desired output dim
Step 2: Use a linear layer to combine head outputs
Step 3: Tesnor shape: (b, num_tokens, d_out)
Step 4: We implicitly split the matrix by adding a num_heads dimension. 
Then we unroll last dim:(b, num_tokens, d_out) -> (b, num_tokens, num_heads, heads_dim)
Step 5: Transpose from shape (b, num_tokens, num_heads, heads_dim) to (b, num_heads, num_tokens, heads_dim)
Step 6 : Compute dot product for each head
Step 7 : Mask truncated to the number of tokens
Step 8 : Use the mask to fill attention scores
Step 9 : Tesnor shape(b, num_tokens, n_heads, head_dim)
Step 10 : Combine heads, where self.d_out = self.num_heads*self.head_dim
Step 11 : Add an optional linear projection
"""

'\nStep 1: Reduce the projection dim to match desired output dim\nStep 2: Use a linear layer to combine head outputs\nStep 3: Tesnor shape: (b, num_tokens, d_out)\nStep 4: We implicitly split the matrix by adding a num_heads dimension. \nThen we unroll last dim:(b, num_tokens, d_out) -> (b, num_tokens, num_heads, heads_dim)\nStep 5: Transpose from shape (b, num_tokens, num_heads, heads_dim) to (b, num_heads, num_tokens, heads_dim)\nStep 6 : Compute dot product for each head\nStep 7 : Mask truncated to the number of tokens\nStep 8 : Use the mask to fill attention scores\nStep 9 : Tesnor shape(b, num_tokens, n_heads, head_dim)\nStep 10 : Combine heads, where self.d_out = self.num_heads*self.head_dim\nStep 11 : Add an optional linear projection\n'

In [13]:
import torch 
torch.manual_seed(123)

inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]
     )
batch = torch.stack((inputs, inputs), dim = 0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads = 2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)



torch.Size([2, 3, 6])
tensor([[[ 0.1569, -0.0873,  0.0210,  0.0215, -0.3243, -0.2518],
         [ 0.1117, -0.0547,  0.0406, -0.0213, -0.3251, -0.2993],
         [ 0.1196, -0.0491,  0.0318, -0.0635, -0.2788, -0.2578]],

        [[ 0.1569, -0.0873,  0.0210,  0.0215, -0.3243, -0.2518],
         [ 0.1117, -0.0547,  0.0406, -0.0213, -0.3251, -0.2993],
         [ 0.1196, -0.0491,  0.0318, -0.0635, -0.2788, -0.2578]]],
       grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 3, 6])
